# 03. Modeling, Evaluation & Explainability

This notebook evaluates predictive models (Random Forest, Gradient Boosting, Logistic Regression) for red wine quality classification and inspects feature importance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay


## 1. Load Processed Datasets


In [ ]:
train_df = pd.read_csv("../data/processed/train_scaled.csv")
test_df = pd.read_csv("../data/processed/test_scaled.csv")

X_train = train_df.drop(columns=["quality_target"])
y_train = train_df["quality_target"]
X_test = test_df.drop(columns=["quality_target"])
y_test = test_df["quality_target"]

print(f"Training instances: {len(X_train)}, Testing instances: {len(X_test)}")


## 2. Train and Benchmark Classifiers


In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    report = classification_report(y_test, y_pred, output_dict=True)
    results[name] = {
        "Accuracy": report["accuracy"],
        "Precision": report["1"]["precision"],
        "Recall": report["1"]["recall"],
        "F1": report["1"]["f1-score"],
        "ROC-AUC": auc
    }

pd.DataFrame(results).T.round(4)


## 3. ROC-AUC Curves Comparison


In [ ]:
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Wine Quality Classification", fontweight="bold")
plt.legend()
plt.show()


## 4. Confusion Matrix (Random Forest Champion Model)


In [ ]:
rf_model = models["RandomForest"]
y_pred_rf = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Standard (<7)", "High Quality (>=7)"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Random Forest", fontweight="bold")
plt.grid(False)
plt.show()


## 5. Feature Importance Analysis


In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Random Forest Feature Importance", fontsize=14, fontweight="bold")
plt.xlabel("Relative Gini Importance")
plt.show()


## 6. Model Serialization


In [ ]:
models_path = Path("../models")
models_path.mkdir(parents=True, exist_ok=True)
joblib.dump(rf_model, models_path / "wine_quality_model.joblib")
print("Champion model serialized to models/wine_quality_model.joblib")
